In [1]:
import torch
import torch.nn.functional as F


In [2]:
#1.导入
x=torch.tensor([[1.0],[2.0],[3.0]])
y=torch.tensor([[0.],[0.],[1.]])

### 基础（单层逻辑回归）

In [3]:
#2.定义模型类
class LogisticRegressionModel(torch.nn.Module):#Module类是所有神经网络模块的基类，对参数进行注册（方便后面的参数管理状态迁移等）
    def __init__(self):
        super(LogisticRegressionModel,self).__init__()
        self.linear=torch.nn.Linear(1,1)#定义神经一层网结构
        
    def forward(self,x):#forward函数定义的是动态的计算图，方便后来的优化器的链式求导
        y_pred=F.sigmoid(self.linear(x))#前馈采用sigmoid输出
        return y_pred
model=LogisticRegressionModel()

## 难度（多层神经层）：

In [4]:
class LogistiRegression(torch.nn.Module):
    def __init__(self):
        super(LogisticRegression,self).__init__()#老生常谈的初始化

        self.layer1=torch.nn.Linear(256,128)
        self.relu=torch.nn.Relu()
        self.layer2=torch.nn.Linear(128,64)
        self.layer3=torch.nn.Linear(64,10) #在init函数内设定各层的结构

    def forward(self,x):
        x=self.layer1(x)
        x=self.relu(x)           #如果不加入非线性的激活函数，整个神经网络等价于一个线性变换，不能拟合非线性函数，所以需要加入非线性的激活函数
        x=self.layer2(x)
        x=self.relu(x)           #激活函数视为独立的一层起作用，其对输入输出没有维度空间的改变，输入什么形状，输出就是什么形状
        x=self.layer3(x)
        return x  #不要忘
        #forward函数在构建动态计算图
        #由于layer123都定义的是线性层，所以都只有一个输入参数
        #由于python的“变量是内存标签”的特性，所以可以只使用一个变量x进行传递
        #由于python的“垃圾回收机制”，旧的内存空间会被收回，所以传递过程是不断将x作为标签贴给新的内存空间
        
  model=LogisticRegression()#定义后立刻实例化    #不要忘     

## 特点：先定义，再调用
### * 不管是各个层还是损失器、优化器的定义与调用过程
### * 定义过程只定义，设置各项设置参数，不涉及计算参数；   调用过程要传入计算参数
### * 在定义的时候已经规定好对于各个计算参数如何处理，调用的时候直接计算不用显式写出

In [14]:
#定义损失函数与优化器
criterion=torch.nn.BCELoss(reduction='mean')#损失函数为二分类交叉熵
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)
#将model中设定的各项参数w1 w2...wn bias传入优化器，使得优化器可以计算loss对于这些参数的偏导数


In [15]:
#开始训练
for epoch in range(1000):
    y_pred = model(x)   #前向传播，启动！   x要与model定义class中的第一层入口维度一致
    loss = criterion(y_pred, y)   #针对已经定义好的代价函数计算代价
    print(epoch, loss.item())

    optimizer.zero_grad()   #清空上一个loop的残余
    loss.backward()    #计算梯度：调用当前代价函数进行计算    沿着类定义中forward函数的计算图反向传播并使用来闹事法则
    optimizer.step()  #步进更新参数：用上一步计算好的梯度进行梯度下降更新各个参数  w-lr*w_grad b-lr*b_grad



0 0.21719872951507568
1 0.21718190610408783
2 0.21716512739658356
3 0.21714837849140167
4 0.217131569981575
5 0.21711480617523193
6 0.21709799766540527
7 0.2170812338590622
8 0.21706454455852509
9 0.21704773604869843
10 0.21703100204467773
11 0.21701423823833466
12 0.21699751913547516
13 0.2169807404279709
14 0.2169639617204666
15 0.21694724261760712
16 0.21693050861358643
17 0.21691377460956573
18 0.21689701080322266
19 0.21688027679920197
20 0.21686355769634247
21 0.21684686839580536
22 0.21683013439178467
23 0.21681340038776398
24 0.21679669618606567
25 0.21678002178668976
26 0.21676327288150787
27 0.21674658358097076
28 0.21672986447811127
29 0.21671314537525177
30 0.21669648587703705
31 0.21667981147766113
32 0.21666306257247925
33 0.21664643287658691
34 0.21662966907024384
35 0.21661299467086792
36 0.2165963500738144
37 0.21657967567443848
38 0.21656303107738495
39 0.21654631197452545
40 0.21652968227863312
41 0.2165130227804184
42 0.21649634838104248
43 0.21647967398166656
44 0.

In [16]:
# 可以查看学习到的权重和偏置（可选）
print("w =", model.linear.weight.item())
print("b =", model.linear.bias.item())

w = 2.259507656097412
b = -5.470348834991455
